# 01 — Juan: BRFSS Data Acquisition & Preprocessing
**Project:** Mental Health Prediction via Dual-Level LSTM | COSC 4337  
**Author:** Juan Perez  
**Deliverables:**
- `brfss_7yr_slim_clean.csv` — individual-level, 7 years stacked, ~60 cols  
- `brfss_state_year_agg_FINAL.csv` — state-year aggregates for Study A LSTM, ~300 rows  
- `brfss_2024_individual.csv` — 2024 only, for Study B individual-level models  

**Run cells in order. Do NOT skip steps.**

---
## CELL 1 — Mount Drive & Install Libraries

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# ── Update this path if Gia's folder is named differently ──
PROJECT_PATH = "/content/drive/MyDrive/BRFSS_Project"

import os
os.makedirs(f"{PROJECT_PATH}/raw_data", exist_ok=True)
os.makedirs(f"{PROJECT_PATH}/clean_data", exist_ok=True)

# pyreadstat is the only non-default library we need
!pip install pyreadstat -q

import pandas as pd
import numpy as np
import pyreadstat
import warnings
warnings.filterwarnings('ignore')

print("Setup complete!")
print(f"Pandas version: {pd.__version__}")
print(f"Project path: {PROJECT_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 13.9 MB/s eta 0:00:00
Setup complete!
Pandas version: 2.2.2
Project path: /content/drive/MyDrive/BRFSS_Project


---
## CELL 2 — Check Available RAM & Drive Space

In [4]:
!cat /proc/meminfo | grep MemTotal
!df -h /content/drive | tail -1
# Free Colab: ~12 GB RAM. Processing one year at a time keeps us well under that.

MemTotal:       13286948 kB
drive           108G   26G   83G  24% /content/drive


---
## CELL 3 — Download BRFSS XPT Files Directly to Drive

> **NOTE:** CDC URLs can change. If a download fails (404), go to  
> https://www.cdc.gov/brfss/annual_data/annual_data.htm  
> download the XPT zip manually, and upload it to `BRFSS_Project/raw_data/`.  
> Then rename it to the convention: `LLCP{YEAR}.XPT`

In [5]:
# CDC direct download URLs — verified pattern as of 2024
urls = {
    "LLCP2024.XPT": "https://www.cdc.gov/brfss/annual_data/2024/files/LLCP2024XPT.zip",
    "LLCP2023.XPT": "https://www.cdc.gov/brfss/annual_data/2023/files/LLCP2023XPT.zip",
    "LLCP2022.XPT": "https://www.cdc.gov/brfss/annual_data/2022/files/LLCP2022XPT.zip",
    "LLCP2021.XPT": "https://www.cdc.gov/brfss/annual_data/2021/files/LLCP2021XPT.zip",
    "LLCP2020.XPT": "https://www.cdc.gov/brfss/annual_data/2020/files/LLCP2020XPT.zip",
    "LLCP2019.XPT": "https://www.cdc.gov/brfss/annual_data/2019/files/LLCP2019XPT.zip",
}

RAW_PATH = f"{PROJECT_PATH}/raw_data"

for filename, url in urls.items():
    zip_path = f"{RAW_PATH}/{filename.replace('.XPT', '.zip')}"
    xpt_path = f"{RAW_PATH}/{filename}"

    if os.path.exists(xpt_path):
        print(f"  ✓ Already exists: {filename}")
        continue

    print(f"Downloading {filename}...")
    result = os.system(f'wget -q "{url}" -O "{zip_path}"')
    if result != 0:
        print(f"  ✗ Download failed for {filename}. Check CDC URL or upload manually.")
        continue

    # Unzip — the XPT inside may have a different name; find and rename it
    os.system(f'unzip -q "{zip_path}" -d "{RAW_PATH}"')
    os.remove(zip_path)

    # Rename any LLCP*.XPT file in raw_data to the standard name
    year = filename.replace('LLCP', '').replace('.XPT', '')
    for f in os.listdir(RAW_PATH):
        if f.upper().startswith('LLCP') and f.upper().endswith('.XPT') and year in f:
            os.rename(f"{RAW_PATH}/{f}", xpt_path)
            break

    print(f"  ✓ Done: {filename}")

print("\nFiles in raw_data:")
for f in sorted(os.listdir(RAW_PATH)):
    size_mb = os.path.getsize(f"{RAW_PATH}/{f}") / 1e6
    print(f"  {f:25s}  {size_mb:.0f} MB")

  ✓ Already exists: LLCP2024.XPT
  ✓ Already exists: LLCP2023.XPT
  ✓ Already exists: LLCP2022.XPT
  ✓ Already exists: LLCP2021.XPT
  ✓ Already exists: LLCP2020.XPT
  ✓ Already exists: LLCP2019.XPT

Files in raw_data:
  LLCP2019.XPT               1137 MB
  LLCP2020.XPT               890 MB
  LLCP2021.XPT               1056 MB
  LLCP2022.XPT               1160 MB
  LLCP2023.XPT               1206 MB
  LLCP2024.XPT               1094 MB


---
## CELL 4 — Define Master Column List
Cross-referenced against Gia's Study A & Study B variable guides.  
Only loading what we need keeps RAM under 500 MB per year.

In [6]:
# ── TARGET ────────────────────────────────────────────────────────────────────
TARGET_COLS = ['MENTHLTH', '_MENT14D', 'PHYSHLTH', 'POORHLTH']

# ── IDENTIFIERS ───────────────────────────────────────────────────────────────
ID_COLS = ['_STATE', 'IYEAR', 'IMONTH', '_LLCPWT']

# ── DEMOGRAPHICS ──────────────────────────────────────────────────────────────
DEMO_COLS = [
    'SEXVAR',    # sex
    '_AGEG5YR',  # age 5-year bands
    '_AGE_G',    # age 6 groups
    '_RACEGR3',  # race/ethnicity
    '_HISPANC',  # hispanic
    'EDUCA', '_EDUCAG',
    'INCOME3', '_INCOMG1',
    'EMPLOY1', 'MARITAL',
    'RENTHOM1', 'VETERAN3',
    'CHILDREN',
    '_METSTAT', '_URBSTAT',
]

# ── ACE VARIABLES ─────────────────────────────────────────────────────────────
# NOTE: ACE module added in 2019. Will be NaN for any pre-2019 data.
ACE_COLS = [
    'ACEDEPRS',  # lived w/ depressed/mentally ill person
    'ACEDRINK',  # lived w/ problem drinker
    'ACEDRUGS',  # lived w/ drug user
    'ACEPRISN',  # lived w/ someone in prison
    'ACEDIVRC',  # parents divorced
    'ACEPUNCH',  # parents beat each other (frequency codes)
    'ACEHURT1',  # parent physically hurt you (frequency codes)
    'ACESWEAR',  # parent swore at you (frequency codes)
    'ACETOUCH',  # touched sexually (frequency codes)
    'ACETTHEM',  # made to touch sexually (frequency codes)
    'ACEHVSEX',  # forced to have sex (frequency codes)
    'ACEADSAF',  # safe/supportive adult (protective; added 2021)
    'ACEADNED',  # basic needs met (protective; added 2021)
]

# ── SOCIAL DETERMINANTS ───────────────────────────────────────────────────────
# NOTE: Most SDOH vars added in 2022. SDLONELY only 2023-2024.
SDOH_COLS = [
    'SDLONELY',  # loneliness (2023+)
    'EMTSUPRT',  # emotional support (2022+)
    'LSATISFY',  # life satisfaction (2022+)
    'SDHEMPLY',  # lost employment (2022+)
    'FOODSTMP',  # food stamps (most years)
    'SDHFOOD1',  # food insecurity (2022+)
    'SDHBILLS',  # unable to pay bills (2022+)
    'SDHUTILS',  # utility bills (2022+)
    'SDHTRNSP',  # transportation barrier (2022+)
    'HOWSAFE1',  # neighborhood safety (2024 only)
    'MEDCOST',   # couldn't afford doctor (2019-2020 name)
    'MEDCOST1',  # couldn't afford doctor (2021+ name)
    'PERSDOC2',  # has personal doctor (2019 name)
    'PERSDOC3',  # has personal doctor (2021+ name)
    'PRIMINS2',  # insurance type (2024 only)
    '_HLTHPL2',  # has health insurance (2024 name)
    '_HCVU654',  # health coverage 18-64 (2024)
]

# ── BEHAVIORAL ────────────────────────────────────────────────────────────────
BEHAV_COLS = [
    'EXERANY2',
    '_TOTINDA',
    '_RFSMOK3',
    '_SMOKER3',
    '_RFBING5',  # binge drinking 2019-2021
    '_RFBING6',  # binge drinking 2022-2024
    '_RFDRHV7',  # heavy alcohol 2019-2021
    '_RFDRHV8',  # heavy alcohol 2022-2023
    '_RFDRHV9',  # heavy alcohol 2024
    'MARIJAN1',
    'ECIGNOW3',  # e-cigarette 2024
    'CHECKUP1',
    'LASTDEN4',
]

# ── PHYSICAL HEALTH & CHRONIC CONDITIONS ─────────────────────────────────────
HEALTH_COLS = [
    'GENHLTH',
    '_BMI5CAT',
    '_RFBMI5',
    'ADDEPEV3',
    'CVDINFR4',
    'CVDCRHD4',
    'CVDSTRK3',
    'DIABETE4',
    'CHCCOPD2',  # COPD 2019-2020 name
    'CHCCOPD3',  # COPD 2021-2024 name
    'HAVARTH4',  # Arthritis (most years)
    'HAVARTH5',  # Arthritis 2021 only
    'CHCKDNY2',
    'ASTHMA3',
    'DEAF', 'BLIND', 'DECIDE', 'DIFFWALK',
    'DIFFDRES', 'DIFFALON',
    'CAREGIV1',
]

# ── COMBINED MASTER LIST ──────────────────────────────────────────────────────
MASTER_COLS = list(dict.fromkeys(
    TARGET_COLS + ID_COLS + DEMO_COLS + ACE_COLS +
    SDOH_COLS + BEHAV_COLS + HEALTH_COLS
))

print(f"Total columns to load per year: {len(MASTER_COLS)}")
print(f"Columns: {MASTER_COLS}")

Total columns to load per year: 88
Columns: ['MENTHLTH', '_MENT14D', 'PHYSHLTH', 'POORHLTH', '_STATE', 'IYEAR', 'IMONTH', '_LLCPWT', 'SEXVAR', '_AGEG5YR', '_AGE_G', '_RACEGR3', '_HISPANC', 'EDUCA', '_EDUCAG', 'INCOME3', '_INCOMG1', 'EMPLOY1', 'MARITAL', 'RENTHOM1', 'VETERAN3', 'CHILDREN', '_METSTAT', '_URBSTAT', 'ACEDEPRS', 'ACEDRINK', 'ACEDRUGS', 'ACEPRISN', 'ACEDIVRC', 'ACEPUNCH', 'ACEHURT1', 'ACESWEAR', 'ACETOUCH', 'ACETTHEM', 'ACEHVSEX', 'ACEADSAF', 'ACEADNED', 'SDLONELY', 'EMTSUPRT', 'LSATISFY', 'SDHEMPLY', 'FOODSTMP', 'SDHFOOD1', 'SDHBILLS', 'SDHUTILS', 'SDHTRNSP', 'HOWSAFE1', 'MEDCOST', 'MEDCOST1', 'PERSDOC2', 'PERSDOC3', 'PRIMINS2', '_HLTHPL2', '_HCVU654', 'EXERANY2', '_TOTINDA', '_RFSMOK3', '_SMOKER3', '_RFBING5', '_RFBING6', '_RFDRHV7', '_RFDRHV8', '_RFDRHV9', 'MARIJAN1', 'ECIGNOW3', 'CHECKUP1', 'LASTDEN4', 'GENHLTH', '_BMI5CAT', '_RFBMI5', 'ADDEPEV3', 'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'DIABETE4', 'CHCCOPD2', 'CHCCOPD3', 'HAVARTH4', 'HAVARTH5', 'CHCKDNY2', 'ASTHMA3', 'DEAF'

---
## CELL 5 — Load & Slim All Years (Memory-Safe)
Uses `pyreadstat` with `usecols` to never load the full 300+ column file.

In [7]:
YEARS = [2019, 2020, 2021, 2022, 2023, 2024]
yearly_dfs = {}

for year in YEARS:
    filepath = f"{PROJECT_PATH}/raw_data/LLCP{year}.XPT"

    if not os.path.exists(filepath):
        print(f"  ✗ File not found: {filepath} — skipping")
        continue

    print(f"\nLoading {year}...")

    try:
        # Peek at column names without loading data
        df_peek, _ = pyreadstat.read_xport(filepath, row_limit=1,
                                            encoding='latin1')
        available_upper = {c.upper(): c for c in df_peek.columns}

        cols_to_load_original = [available_upper[c] for c in MASTER_COLS if c in available_upper]
        missing = [c for c in MASTER_COLS if c not in available_upper]

        if missing:
            print(f"  ⚠ {len(missing)} cols not in {year} (expected for some years):")
            print(f"    {missing}")

        # Load only the columns we need
        df, _ = pyreadstat.read_xport(filepath, usecols=cols_to_load_original,
                                       encoding='latin1')
        df.columns = df.columns.str.upper()
        df['SURVEY_YEAR'] = year

        yearly_dfs[year] = df
        print(f"  ✓ Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
        print(f"    Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

    except Exception as e:
        print(f"  ✗ Failed to load {year}: {e}")

print(f"\nAll available years loaded!")


Loading 2019...
  ⚠ 25 cols not in 2019 (expected for some years):
    ['INCOME3', '_INCOMG1', 'ACEADSAF', 'ACEADNED', 'SDLONELY', 'EMTSUPRT', 'LSATISFY', 'SDHEMPLY', 'SDHFOOD1', 'SDHBILLS', 'SDHUTILS', 'SDHTRNSP', 'HOWSAFE1', 'MEDCOST1', 'PERSDOC3', 'PRIMINS2', '_HLTHPL2', '_HCVU654', '_RFBING6', '_RFDRHV8', '_RFDRHV9', 'ECIGNOW3', 'LASTDEN4', 'CHCCOPD3', 'HAVARTH5']
  ✓ Loaded: 418,268 rows × 64 cols
    Memory: 251.0 MB

Loading 2020...
  ⚠ 25 cols not in 2020 (expected for some years):
    ['INCOME3', '_INCOMG1', 'ACEADSAF', 'ACEADNED', 'SDLONELY', 'EMTSUPRT', 'LSATISFY', 'SDHEMPLY', 'FOODSTMP', 'SDHFOOD1', 'SDHBILLS', 'SDHUTILS', 'SDHTRNSP', 'HOWSAFE1', 'MEDCOST1', 'PERSDOC3', 'PRIMINS2', '_HLTHPL2', '_HCVU654', '_RFBING6', '_RFDRHV8', '_RFDRHV9', 'ECIGNOW3', 'CHCCOPD3', 'HAVARTH5']
  ✓ Loaded: 401,958 rows × 64 cols
    Memory: 241.2 MB

Loading 2021...
  ⚠ 22 cols not in 2021 (expected for some years):
    ['SDLONELY', 'EMTSUPRT', 'LSATISFY', 'SDHEMPLY', 'FOODSTMP', 'SDHFOOD1',

---
## CELL 6 — Stack All Years

In [8]:
df_all = pd.concat(yearly_dfs.values(), ignore_index=True, sort=False)

print(f"Combined shape: {df_all.shape}")
print(f"Total memory: {df_all.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print("\nRow counts per year:")
print(df_all['SURVEY_YEAR'].value_counts().sort_index())

Combined shape: (2595044, 89)
Total memory: 2076.0 MB

Row counts per year:
SURVEY_YEAR
2019    418268
2020    401958
2021    438693
2022    445132
2023    433323
2024    457670
Name: count, dtype: int64


---
## CELL 7 — Recode Missing Values
BRFSS encodes missing/refused/don't know as 7, 9, 77, 99, 777, 999, etc.  
88 = "None / Zero" — this is VALID, not missing.

In [9]:
def recode_missing(df):
    """
    Recode BRFSS 'don't know / refused' codes to NaN.
    88 / 888 = None / Zero — VALID, do NOT recode to NaN.
    """
    df = df.copy()

    # Variable-specific missing codes
    missing_map = {
        # Binary Yes/No vars: 7=DK, 9=Refused
        'EXERANY2':  [7, 9],
        'CVDINFR4':  [7, 9],
        'CVDCRHD4':  [7, 9],
        'CVDSTRK3':  [7, 9],
        'ASTHMA3':   [7, 9],
        'ADDEPEV3':  [7, 9],
        'CHCCOPD2':  [7, 9],
        'CHCCOPD3':  [7, 9],
        'HAVARTH4':  [7, 9],
        'HAVARTH5':  [7, 9],
        'CHCKDNY2':  [7, 9],
        'DEAF':      [7, 9],
        'BLIND':     [7, 9],
        'DECIDE':    [7, 9],
        'DIFFWALK':  [7, 9],
        'DIFFDRES':  [7, 9],
        'DIFFALON':  [7, 9],
        'CAREGIV1':  [7, 9],
        'MEDCOST':   [7, 9],
        'MEDCOST1':  [7, 9],
        'VETERAN3':  [7, 9],
        'RENTHOM1':  [7, 9],
        'FOODSTMP':  [7, 9],
        'SDHBILLS':  [7, 9],
        'SDHUTILS':  [7, 9],
        'SDHTRNSP':  [7, 9],
        'SDHEMPLY':  [7, 9],
        'SDHFOOD1':  [7, 9],
        'SDLONELY':  [7, 9],
        'EMTSUPRT':  [7, 9],
        'LSATISFY':  [7, 9],
        'HOWSAFE1':  [7, 9],
        # ACE vars
        'ACEDEPRS':  [7, 9],
        'ACEDRINK':  [7, 9],
        'ACEDRUGS':  [7, 9],
        'ACEPRISN':  [7, 9],
        'ACEDIVRC':  [7, 9],
        'ACEPUNCH':  [7, 9],  # frequency codes, but 7/9 still = missing
        'ACEHURT1':  [7, 9],
        'ACESWEAR':  [7, 9],
        'ACETOUCH':  [7, 9],
        'ACETTHEM':  [7, 9],
        'ACEHVSEX':  [7, 9],
        'ACEADSAF':  [7, 9],
        'ACEADNED':  [7, 9],
        # Multi-code
        'GENHLTH':   [7, 9],
        'CHECKUP1':  [7, 9],   # NOTE: 8=Never is VALID here
        'LASTDEN4':  [7, 9],
        'EMPLOY1':   [9],
        'MARITAL':   [9],
        'EDUCA':     [9],
        'INCOME3':   [77, 99],
        'CHILDREN':  [99],
        'MENTHLTH':  [77, 99],  # 88=None(0 days) — handled below
        'PHYSHLTH':  [77, 99],
        'POORHLTH':  [77, 99],
        'MARIJAN1':  [77, 99],
        'PERSDOC2':  [7, 9],
        'PERSDOC3':  [7, 9],
    }

    for col, codes in missing_map.items():
        if col in df.columns:
            df[col] = df[col].replace(codes, np.nan)

    # MENTHLTH/PHYSHLTH: 88 means "None" (0 bad days) — replace with 0
    for col in ['MENTHLTH', 'PHYSHLTH', 'POORHLTH']:
        if col in df.columns:
            df[col] = df[col].replace(88, 0)

    # MARIJAN1: 88 = None (0 days)
    if 'MARIJAN1' in df.columns:
        df['MARIJAN1'] = df['MARIJAN1'].replace(88, 0)

    return df

df_all = recode_missing(df_all)
print("Missing value recoding complete.")
print(f"\nMissing % for MENTHLTH: {df_all['MENTHLTH'].isna().mean()*100:.2f}%")

Missing value recoding complete.

Missing % for MENTHLTH: 1.93%


---
## CELL 8 — Recode Binary Yes/No Variables to 0/1
BRFSS standard: 1=Yes → 1, 2=No → 0

In [10]:

def recode_binary(df):
    """Recode standard Yes/No variables: 1=Yes→1, 2=No→0."""
    df = df.copy()

    binary_vars = [
        'EXERANY2', 'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'ASTHMA3',
        'ADDEPEV3', 'CHCCOPD2', 'CHCCOPD3', 'HAVARTH4', 'HAVARTH5',
        'CHCKDNY2', 'DEAF', 'BLIND', 'DECIDE', 'DIFFWALK',
        'DIFFDRES', 'DIFFALON', 'CAREGIV1', 'MEDCOST', 'MEDCOST1',
        'VETERAN3', 'FOODSTMP', 'SDHBILLS', 'SDHUTILS',
        'SDHTRNSP', 'SDHEMPLY',
        # ACE binary items (1=Yes, 2=No)
        'ACEDEPRS', 'ACEDRINK', 'ACEDRUGS', 'ACEPRISN', 'ACEDIVRC',
        'ACEADSAF', 'ACEADNED',
    ]

    for col in binary_vars:
        if col in df.columns:
            df[col] = df[col].map({1: 1, 2: 0})

    # ── DIABETE4 special: 5 categories ──────────────────────────────────────
    # 1=Yes, 2=Yes(gestational only)→NaN, 3=No, 4=Pre-diabetes→0, 7/9→NaN
    if 'DIABETE4' in df.columns:
        df['DIABETE4'] = df['DIABETE4'].apply(
            lambda x: 1 if x == 1 else (0 if x in [3, 4] else np.nan)
        )

    # ── ACE frequency variables: 1=Never→0, 2=Once→1, 3=More than once→1 ───
    ace_freq_cols = ['ACEPUNCH', 'ACEHURT1', 'ACESWEAR', 'ACETOUCH', 'ACETTHEM', 'ACEHVSEX']
    for col in ace_freq_cols:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: 0 if x == 1 else (1 if x in [2, 3] else np.nan)
            )

    return df

df_all = recode_binary(df_all)
print("Binary recoding complete.")
print(f"Sample — ADDEPEV3 value counts: {df_all['ADDEPEV3'].value_counts().to_dict()}")

Binary recoding complete.
Sample — ADDEPEV3 value counts: {0.0: 2063995, 1.0: 516083}


---
## CELL 9 — Harmonize Variables With Name Changes Across Years
Merges renamed variables into single unified columns. Required before aggregation.

In [11]:
def harmonize_variables(df):
    """Merge BRFSS variables that were renamed across survey years."""
    df = df.copy()

    # ── BINGE DRINKING ───────────────────────────────────────────────────────
    # _RFBING5 (2019-2021), _RFBING6 (2022-2024)
    for col in ['_RFBING5', '_RFBING6']:
        if col not in df.columns:
            df[col] = np.nan
    df['BINGE_DRINK'] = df['_RFBING5'].combine_first(df['_RFBING6'])
    df['BINGE_DRINK'] = df['BINGE_DRINK'].map({1: 0, 2: 1})

    # ── HEAVY DRINKING ───────────────────────────────────────────────────────
    # THREE names: _RFDRHV7 (2019-2021), _RFDRHV8 (2022-2023), _RFDRHV9 (2024)
    for col in ['_RFDRHV7', '_RFDRHV8', '_RFDRHV9']:
        if col not in df.columns:
            df[col] = np.nan
    df['HEAVY_DRINK'] = (
        df['_RFDRHV7'].combine_first(df['_RFDRHV8']).combine_first(df['_RFDRHV9'])
    )
    df['HEAVY_DRINK'] = df['HEAVY_DRINK'].map({1: 0, 2: 1})

    # ── COPD ─────────────────────────────────────────────────────────────────
    # CHCCOPD2 (2019-2020), CHCCOPD3 (2021-2024) — already recoded to 0/1
    for col in ['CHCCOPD2', 'CHCCOPD3']:
        if col not in df.columns:
            df[col] = np.nan
    df['COPD'] = df['CHCCOPD2'].combine_first(df['CHCCOPD3'])

    # ── ARTHRITIS ────────────────────────────────────────────────────────────
    # HAVARTH4 (all except 2021), HAVARTH5 (2021 only)
    for col in ['HAVARTH4', 'HAVARTH5']:
        if col not in df.columns:
            df[col] = np.nan
    df['ARTHRITIS'] = df['HAVARTH4'].combine_first(df['HAVARTH5'])

    # ── COULDN'T AFFORD DOCTOR ───────────────────────────────────────────────
    # MEDCOST (2019-2020), MEDCOST1 (2021-2024)
    for col in ['MEDCOST', 'MEDCOST1']:
        if col not in df.columns:
            df[col] = np.nan
    df['NO_AFFORD_DR'] = df['MEDCOST'].combine_first(df['MEDCOST1'])

    # ── HAS PERSONAL DOCTOR ──────────────────────────────────────────────────
    # PERSDOC2 (2019), PERSDOC3 (2021-2024), missing 2020 → NaN
    for col in ['PERSDOC2', 'PERSDOC3']:
        if col not in df.columns:
            df[col] = np.nan
    df['HAS_DOCTOR'] = df['PERSDOC2'].combine_first(df['PERSDOC3'])
    df['HAS_DOCTOR'] = df['HAS_DOCTOR'].apply(
        lambda x: 1 if x in [1, 2] else (0 if x == 3 else np.nan)
    )

    # ── FREQUENT DISTRESS TARGET ─────────────────────────────────────────────
    # _MENT14D: 1=14+ bad days, 2=fewer than 14
    if '_MENT14D' in df.columns:
        df['FREQ_DISTRESS'] = df['_MENT14D'].map({1: 0, 2: 0, 3: 1})

    # ── POOR GENERAL HEALTH ──────────────────────────────────────────────────
    # GENHLTH: 1=Excellent→5=Poor; 4 or 5 = Fair/Poor
    if 'GENHLTH' in df.columns:
        df['POOR_GENHLTH'] = df['GENHLTH'].apply(
            lambda x: 1 if x in [4, 5] else (0 if x in [1, 2, 3] else np.nan)
        )

    # ── PHYSICAL INACTIVITY ──────────────────────────────────────────────────
    # _TOTINDA: 1=Had activity, 2=No activity → invert to 0/1 inactive
    if '_TOTINDA' in df.columns:
        df['PHYS_INACTIVE'] = df['_TOTINDA'].map({1: 0, 2: 1})

    # ── CURRENT SMOKER ───────────────────────────────────────────────────────
    # _RFSMOK3: 1=Current, 2=Not current
    if '_RFSMOK3' in df.columns:
        df['CURRENT_SMOKER'] = df['_RFSMOK3'].map({1: 0, 2: 1})

    # ── OVERWEIGHT/OBESE ─────────────────────────────────────────────────────
    if '_RFBMI5' in df.columns:
        df['OVERWEIGHT_OBESE'] = df['_RFBMI5'].map({1: 0, 2: 1})

    # ── FOOD STAMPS ──────────────────────────────────────────────────────────
    # Already binary 0/1 from recode_binary; just rename for clarity
    if 'FOODSTMP' in df.columns:
        df['FOOD_STAMPS'] = df['FOODSTMP']

    # ── DEPRESSION DIAGNOSIS ─────────────────────────────────────────────────
    if 'ADDEPEV3' in df.columns:
        df['DEPRESSION_DX'] = df['ADDEPEV3']  # already 0/1

    # ── DIABETES ─────────────────────────────────────────────────────────────
    if 'DIABETE4' in df.columns:
        df['DIABETES'] = df['DIABETE4']  # already recoded in recode_binary

    # ── HEART ATTACK & STROKE ────────────────────────────────────────────────
    if 'CVDINFR4' in df.columns:
        df['HEART_ATTACK'] = df['CVDINFR4']
    if 'CVDSTRK3' in df.columns:
        df['STROKE'] = df['CVDSTRK3']

    # ── LAST CHECKUP ─────────────────────────────────────────────────────────
    # 1=Past year, 2=1-2 years, 3=2-5 years, 4=5+ years, 8=Never
    if 'CHECKUP1' in df.columns:
        df['CHECKUP_RECENT'] = df['CHECKUP1'].apply(
            lambda x: 1 if x == 1 else (0 if x in [2, 3, 4, 8] else np.nan)
        )

    # ── SDOH HARDSHIP (2022-2024 only, NaN for earlier years) ────────────────
    hardship_cols = [c for c in ['SDHBILLS', 'SDHUTILS', 'SDHEMPLY'] if c in df.columns]
    if hardship_cols:
        df['HARDSHIP_ANY'] = df[hardship_cols].max(axis=1)

    if 'SDHFOOD1' in df.columns:
        # 1=Always, 2=Usually, 3=Sometimes food insecure; 4=Rarely, 5=Never = secure
        df['FOOD_INSECURE'] = df['SDHFOOD1'].apply(
            lambda x: 1 if x in [1, 2, 3] else (0 if x in [4, 5] else np.nan)
        )

    # ── EMOTIONAL SUPPORT & LIFE SATISFACTION (2022-2024) ────────────────────
    if 'EMTSUPRT' in df.columns:
        # 1=Always...5=Never; high score = low support
        df['LOW_EMOTIONAL_SUPPORT'] = df['EMTSUPRT'].apply(
            lambda x: 1 if x in [4, 5] else (0 if x in [1, 2, 3] else np.nan)
        )

    if 'LSATISFY' in df.columns:
        # 1=Very satisfied, 2=Satisfied, 3=Dissatisfied, 4=Very dissatisfied
        df['LIFE_DISSATISFIED'] = df['LSATISFY'].apply(
            lambda x: 1 if x in [3, 4] else (0 if x in [1, 2] else np.nan)
        )

    # ── ACE COMPOSITE SCORE ──────────────────────────────────────────────────
    ace_adversity = [
        'ACEDEPRS', 'ACEDRINK', 'ACEDRUGS', 'ACEPRISN', 'ACEDIVRC',
        'ACEPUNCH', 'ACEHURT1', 'ACESWEAR', 'ACETOUCH', 'ACETTHEM', 'ACEHVSEX'
    ]
    ace_available = [c for c in ace_adversity if c in df.columns]
    if ace_available:
        df['ACE_SCORE'] = df[ace_available].sum(axis=1, skipna=True)
        # BUG FIX: pandas sum(skipna=True) returns 0 for all-NaN rows.
        # This causes states/years that never asked ACE questions to show
        # ACE_SCORE = 0 instead of NaN, corrupting state-year aggregates.
        # Fix: explicitly set ACE_SCORE to NaN when ALL items were missing.
        all_nan_mask = df[ace_available].isna().all(axis=1)
        df.loc[all_nan_mask, 'ACE_SCORE'] = np.nan
        pct_with_ace = (~all_nan_mask).mean() * 100
        print(f"  ACE_SCORE: {pct_with_ace:.1f}% of respondents answered ACE module")
        df['HIGH_ACE'] = (df['ACE_SCORE'] >= 4).astype(float)


    # Household dysfunction composite
    hh_cols = [c for c in ['ACEDEPRS', 'ACEDRINK', 'ACEDRUGS'] if c in df.columns]
    if hh_cols:
        df['ACE_HH_DYSFUNCTION'] = df[hh_cols].max(axis=1)

    # Protective ACE factors (2021+)
    if 'ACEADSAF' in df.columns:
        df['ACEADSAF_BIN'] = df['ACEADSAF']  # already 0/1
    if 'ACEADNED' in df.columns:
        df['ACEADNED_BIN'] = df['ACEADNED']

    # ── PRINT SUMMARY ────────────────────────────────────────────────────────
    new_cols = [
        'BINGE_DRINK', 'HEAVY_DRINK', 'COPD', 'ARTHRITIS', 'NO_AFFORD_DR',
        'HAS_DOCTOR', 'FREQ_DISTRESS', 'POOR_GENHLTH', 'PHYS_INACTIVE',
        'CURRENT_SMOKER', 'OVERWEIGHT_OBESE', 'FOOD_STAMPS', 'DEPRESSION_DX',
        'DIABETES', 'HEART_ATTACK', 'STROKE', 'CHECKUP_RECENT',
        'ACE_SCORE', 'ACE_HH_DYSFUNCTION', 'HARDSHIP_ANY', 'FOOD_INSECURE',
        'LOW_EMOTIONAL_SUPPORT', 'LIFE_DISSATISFIED',
    ]
    print("Harmonization complete. New columns:")
    for c in new_cols:
        status = '✓' if c in df.columns else '✗ MISSING'
        print(f"  {status} — {c}")

    return df

df_all = harmonize_variables(df_all)

  ACE_SCORE: 16.8% of respondents answered ACE module
Harmonization complete. New columns:
  ✓ — BINGE_DRINK
  ✓ — HEAVY_DRINK
  ✓ — COPD
  ✓ — ARTHRITIS
  ✓ — NO_AFFORD_DR
  ✓ — HAS_DOCTOR
  ✓ — FREQ_DISTRESS
  ✓ — POOR_GENHLTH
  ✓ — PHYS_INACTIVE
  ✓ — CURRENT_SMOKER
  ✓ — OVERWEIGHT_OBESE
  ✓ — FOOD_STAMPS
  ✓ — DEPRESSION_DX
  ✓ — DIABETES
  ✓ — HEART_ATTACK
  ✓ — STROKE
  ✓ — CHECKUP_RECENT
  ✓ — ACE_SCORE
  ✓ — ACE_HH_DYSFUNCTION
  ✓ — HARDSHIP_ANY
  ✓ — FOOD_INSECURE
  ✓ — LOW_EMOTIONAL_SUPPORT
  ✓ — LIFE_DISSATISFIED


---
## CELL 10 — Engineer Features

In [12]:
def engineer_features(df):
    df = df.copy()

    # 1. Binary TARGET: frequent mental distress (14+ bad days)
    if 'MENTHLTH' in df.columns:
        df['FREQUENT_DISTRESS'] = np.where(
            df['MENTHLTH'].isna(), np.nan,
            np.where(df['MENTHLTH'] >= 14, 1, 0)
        )
        print(f"Target prevalence: {df['FREQUENT_DISTRESS'].mean()*100:.1f}%")

    # 2. Economic hardship index (0-4)
    hardship_items = ['SDHBILLS', 'SDHUTILS', 'SDHTRNSP', 'SDHFOOD1']
    hardship_available = [c for c in hardship_items if c in df.columns]
    if hardship_available:
        df['HARDSHIP_INDEX'] = df[hardship_available].sum(axis=1, skipna=True)

    # 3. Physical multimorbidity (count of chronic conditions)
    chronic_items = ['CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'COPD', 'ARTHRITIS',
                     'CHCKDNY2', 'ASTHMA3', 'DIABETE4']
    chronic_available = [c for c in chronic_items if c in df.columns]
    if chronic_available:
        df['MULTIMORBIDITY'] = df[chronic_available].sum(axis=1, skipna=True)

    # 4. Any physical disability flag
    disability_items = ['DEAF', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON']
    disability_available = [c for c in disability_items if c in df.columns]
    if disability_available:
        df['ANY_DISABILITY'] = (
            df[disability_available].sum(axis=1, skipna=True) > 0
        ).astype(float)

    return df

df_all = engineer_features(df_all)
print("\nFeature engineering complete.")

Target prevalence: 13.1%

Feature engineering complete.


---
## CELL 11 — Data Quality Report
Share the output of this cell with Gia as your Task 4 check-in.

In [13]:
print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)
print(f"\nTotal rows: {len(df_all):,}")
print(f"Total columns: {df_all.shape[1]}")
print(f"Years: {sorted(df_all['SURVEY_YEAR'].unique())}")

print(f"\nRows per year:")
print(df_all['SURVEY_YEAR'].value_counts().sort_index().to_string())

print(f"\nTarget variable (FREQUENT_DISTRESS):")
if 'FREQUENT_DISTRESS' in df_all.columns:
    print(f"  Class 0 (no distress): {(df_all['FREQUENT_DISTRESS']==0).sum():,}")
    print(f"  Class 1 (frequent):    {(df_all['FREQUENT_DISTRESS']==1).sum():,}")
    print(f"  Missing:               {df_all['FREQUENT_DISTRESS'].isna().sum():,}")
    print(f"  Prevalence:            {df_all['FREQUENT_DISTRESS'].mean()*100:.1f}%")

print(f"\nTop 15 columns by % missing:")
missing_pct = (df_all.isna().mean() * 100).sort_values(ascending=False).head(15)
for col, pct in missing_pct.items():
    print(f"  {col:25s}: {pct:.1f}%")

print(f"\nKey harmonized columns — missing % by year:")
key_cols = ['BINGE_DRINK', 'HEAVY_DRINK', 'COPD', 'HARDSHIP_ANY', 'ACE_SCORE']
for col in key_cols:
    if col in df_all.columns:
        by_year = df_all.groupby('SURVEY_YEAR')[col].apply(lambda x: x.isna().mean()*100)
        print(f"  {col}: {by_year.round(1).to_dict()}")

DATA QUALITY REPORT

Total rows: 2,595,044
Total columns: 119
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Rows per year:
SURVEY_YEAR
2019    418268
2020    401958
2021    438693
2022    445132
2023    433323
2024    457670

Target variable (FREQUENT_DISTRESS):
  Class 0 (no distress): 2,211,143
  Class 1 (frequent):    333,945
  Missing:               49,956
  Prevalence:            13.1%

Top 15 columns by % missing:
  ACEADNED                 : 99.8%
  ACEADNED_BIN             : 99.8%
  ACEADSAF_BIN             : 99.5%
  ACEADSAF                 : 99.5%
  HOWSAFE1                 : 92.3%
  ACEDIVRC                 : 83.6%
  ACESWEAR                 : 83.6%
  ACEPUNCH                 : 83.6%
  ACEHVSEX                 : 83.6%
  ACETOUCH                 : 83.6%
  ACETTHEM                 : 83.6%
  SDLONELY                 : 83.6%
  ECIGNOW3                 : 83.5%
  ACEHURT1                 : 83.5%
  ACEDEPRS                 

---
## CELL 12 — Build State-Year Aggregate Table (Study A LSTM)
Collapses individual rows to one row per state per year.  
Expected output: ~300 rows (50 states × 6 years).

In [14]:
# Final feature list for aggregation — matches Gia's Study A FINAL guide
FINAL_AGG_FEATURES = [
    # Targets
    'FREQ_DISTRESS', 'MENTHLTH', 'PHYSHLTH',
    # Physical health
    'POOR_GENHLTH',
    # Behavioral
    'PHYS_INACTIVE', 'CURRENT_SMOKER', 'BINGE_DRINK', 'HEAVY_DRINK',
    'OVERWEIGHT_OBESE', 'MARIJAN1',
    # Chronic conditions
    'DEPRESSION_DX', 'DIABETES', 'HEART_ATTACK', 'STROKE', 'COPD', 'ARTHRITIS',
    # Healthcare access
    'NO_AFFORD_DR', 'CHECKUP_RECENT', 'HAS_DOCTOR',
    # Food/economic (NaN for 2019-2021 — structural, not a bug)
    'FOOD_STAMPS', 'HARDSHIP_ANY', 'FOOD_INSECURE',
    # Social (NaN for 2019-2021)
    'LOW_EMOTIONAL_SUPPORT', 'LIFE_DISSATISFIED',
    # ACE
    'ACE_SCORE', 'ACE_HH_DYSFUNCTION',
    # ACE protective (NaN for 2019-2020)
    'ACEADSAF_BIN', 'ACEADNED_BIN',
]

# Keep only features that exist in df_all
available_feats = [f for f in FINAL_AGG_FEATURES if f in df_all.columns]
missing_feats = [f for f in FINAL_AGG_FEATURES if f not in df_all.columns]
if missing_feats:
    print(f"⚠ Features not found (check harmonization): {missing_feats}")

# Aggregate: mean = prevalence rate for 0/1 vars, average for continuous
state_year_agg = (
    df_all
    .groupby(['_STATE', 'SURVEY_YEAR'])[available_feats]
    .mean()
    .reset_index()
)

# Add sample size per cell (useful for LSTM weighting / QC)
n_per_cell = df_all.groupby(['_STATE', 'SURVEY_YEAR']).size().reset_index(name='N_RESPONDENTS')
state_year_agg = state_year_agg.merge(n_per_cell, on=['_STATE', 'SURVEY_YEAR'])

print(f"\nFinal state-year aggregate shape: {state_year_agg.shape}")
print(f"Expected: ~300 rows (50 states × 6 years) × ~28 feature columns")
print(f"\nMissing % by feature (NaN = structural absence, not an error):")
missing_by_feat = (state_year_agg.isna().mean() * 100).sort_values(ascending=False)
print(missing_by_feat[missing_by_feat > 0].round(1).to_string())

print(f"\nPreview:")
print(state_year_agg.head())


Final state-year aggregate shape: (317, 31)
Expected: ~300 rows (50 states × 6 years) × ~28 feature columns

Missing % by feature (NaN = structural absence, not an error):
ACEADSAF_BIN             88.3
ACEADNED_BIN             88.3
ACE_HH_DYSFUNCTION       76.0
ACE_SCORE                76.0
LOW_EMOTIONAL_SUPPORT    66.9
FOOD_INSECURE            66.9
HARDSHIP_ANY             66.9
LIFE_DISSATISFIED        66.9
FOOD_STAMPS              65.6
MARIJAN1                 64.4

Preview:
   _STATE  SURVEY_YEAR  FREQ_DISTRESS  MENTHLTH  PHYSHLTH  POOR_GENHLTH  \
0     1.0         2019       0.150246  4.745738  5.436203      0.242144   
1     1.0         2020       0.144108  4.488292  4.190321      0.218885   
2     1.0         2021       0.156181  4.787328  4.582501      0.217923   
3     1.0         2022       0.156916  4.828118  5.065967      0.248498   
4     1.0         2023       0.141094  4.440047  4.745263      0.247875   

   PHYS_INACTIVE  CURRENT_SMOKER  BINGE_DRINK  HEAVY_DRINK  ...  F

---
## CELL 13 — Save All Three Output Files to Shared Drive
Ping Gia in group chat when all three are saved.

In [15]:
CLEAN_PATH = f"{PROJECT_PATH}/clean_data"

# ── File 1: Full 6-year individual-level slim dataset ─────────────────────────
output1 = f"{CLEAN_PATH}/brfss_7yr_slim_clean.csv"
df_all.to_csv(output1, index=False)
size1 = os.path.getsize(output1) / 1e6
print(f"✓ Saved: brfss_7yr_slim_clean.csv ({len(df_all):,} rows, {size1:.0f} MB)")

# ── File 2: State-year aggregates for Study A LSTM ────────────────────────────
output2 = f"{CLEAN_PATH}/brfss_state_year_agg_FINAL.csv"
state_year_agg.to_csv(output2, index=False)
size2 = os.path.getsize(output2) / 1e6
print(f"✓ Saved: brfss_state_year_agg_FINAL.csv ({len(state_year_agg):,} rows, {size2:.1f} MB)")

# ── File 3: 2024 only — for Study B individual-level models ──────────────────
df_2024 = df_all[df_all['SURVEY_YEAR'] == 2024].copy()
output3 = f"{CLEAN_PATH}/brfss_2024_individual.csv"
df_2024.to_csv(output3, index=False)
size3 = os.path.getsize(output3) / 1e6
print(f"✓ Saved: brfss_2024_individual.csv ({len(df_2024):,} rows, {size3:.0f} MB)")

print("\n✅ All files saved to shared Drive!")
print("   → Ping Gia with the shape outputs from Cells 11 and 12.")

✓ Saved: brfss_7yr_slim_clean.csv (2,595,044 rows, 963 MB)
✓ Saved: brfss_state_year_agg_FINAL.csv (317 rows, 0.1 MB)
✓ Saved: brfss_2024_individual.csv (457,670 rows, 179 MB)

✅ All files saved to shared Drive!
   → Ping Gia with the shape outputs from Cells 11 and 12.


---
## CELL 14 — Deliverable Checklist
Run this cell last as a final sanity check before messaging the team.

In [16]:
print("=" * 55)
print("DELIVERABLE CHECKLIST")
print("=" * 55)

checks = {
    "Task 1 — Colab + Drive mounted": True,  # you're here
    "Task 2 — XPT files downloaded": all(
        os.path.exists(f"{PROJECT_PATH}/raw_data/LLCP{y}.XPT")
        for y in [2019, 2020, 2021, 2022, 2023, 2024]
    ),
    "Task 3 — Years loaded": 'df_all' in dir() and len(yearly_dfs) >= 5,
    "Task 4 — Cleaning complete": 'FREQUENT_DISTRESS' in df_all.columns,
    "Task 5 — State-year agg built (~300 rows)": (
        'state_year_agg' in dir() and len(state_year_agg) >= 250
    ),
    "Task 6a — brfss_7yr_slim_clean.csv saved": os.path.exists(
        f"{PROJECT_PATH}/clean_data/brfss_7yr_slim_clean.csv"
    ),
    "Task 6b — brfss_state_year_agg_FINAL.csv saved": os.path.exists(
        f"{PROJECT_PATH}/clean_data/brfss_state_year_agg_FINAL.csv"
    ),
    "Task 6c — brfss_2024_individual.csv saved": os.path.exists(
        f"{PROJECT_PATH}/clean_data/brfss_2024_individual.csv"
    ),
}

all_done = True
for task, status in checks.items():
    icon = "✅" if status else "❌"
    print(f"  {icon}  {task}")
    if not status:
        all_done = False

print()
if all_done:
    print("🎉 All tasks complete! Message Gia and share Cell 11 + Cell 12 outputs.")
else:
    print("⚠  Some tasks incomplete — review the cells above for errors.")

DELIVERABLE CHECKLIST
  ✅  Task 1 — Colab + Drive mounted
  ✅  Task 2 — XPT files downloaded
  ✅  Task 3 — Years loaded
  ✅  Task 4 — Cleaning complete
  ✅  Task 5 — State-year agg built (~300 rows)
  ✅  Task 6a — brfss_7yr_slim_clean.csv saved
  ✅  Task 6b — brfss_state_year_agg_FINAL.csv saved
  ✅  Task 6c — brfss_2024_individual.csv saved

🎉 All tasks complete! Message Gia and share Cell 11 + Cell 12 outputs.


In [17]:


print(f"Loaded: {df_all.shape}")
print(f"Columns: {list(df_all.columns[:10])} ...")
print(f"Years: {sorted(df_all['SURVEY_YEAR'].unique())}")
# DIAGNOSTIC — run this and send Gia the output
print("_MENT14D unique values and counts:")
print(df_all['_MENT14D'].value_counts().sort_index())

print("\n_RFBING6 unique values:")
print(df_all['_RFBING6'].value_counts().sort_index())

print("\nFREQ_DISTRESS after harmonize (should be ~0.13-0.15):")
print(df_all['FREQ_DISTRESS'].mean())

print("\nBINGE_DRINK after harmonize (should be ~0.17):")
print(df_all['BINGE_DRINK'].mean())

Loaded: (2595044, 119)
Columns: ['_STATE', 'IMONTH', 'IYEAR', 'SEXVAR', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'PERSDOC2', 'MEDCOST'] ...
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
_MENT14D unique values and counts:
_MENT14D
1.0    1586369
2.0     624774
3.0     333945
9.0      49956
Name: count, dtype: int64

_RFBING6 unique values:
_RFBING6
1.0    1039221
2.0     164977
9.0     131927
Name: count, dtype: int64

FREQ_DISTRESS after harmonize (should be ~0.13-0.15):
0.13121157303794603

BINGE_DRINK after harmonize (should be ~0.17):
0.13508887336990896
